# Build the visits-level KPI table

Aggregates the event-level data into a visit-level table where each row represents one visit. Computes per-visit KPIs that will later feed into client-level metrics and hypothesis testing.

**Per-visit KPIs computed**
- `max_step_reached` — furthest step the user reached
- `n_steps` — total number of events in the visit
- `n_backward_steps` — number of backward navigations
- `error_rate` — fraction of events that were backward
- `reached_confirm` — whether the visit completed the funnel
- `visit_duration_sec` — total visit duration in seconds

In [2]:
## Load events table
import pandas as pd
df_events = pd.read_csv("events_kpi_table.csv", parse_dates=["date_time"])

In [3]:
df_events

,client_id,visitor_id,visit_id,process_step,date_time,step_num,previous_step_num,step_direction,time_on_each_step
0,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,0,NaN,first_visit,NaN
1,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,1,0.0,forward,9.0
2,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,2,1.0,forward,46.0
3,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,3,2.0,forward,94.0
4,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,4,3.0,forward,64.0
...,...,...,...,...,...,...,...,...,...
755400,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:40:08,1,0.0,forward,7.0
755401,9999875,738878760_1556639849,931268933_219402947_599432,step_1,2017-06-01 22:41:28,1,1.0,repeat,80.0
755402,9999875,738878760_1556639849,931268933_219402947_599432,step_2,2017-06-01 22:41:47,2,1.0,forward,19.0
755403,9999875,738878760_1556639849,931268933_219402947_599432,step_3,2017-06-01 22:44:58,3,2.0,forward,191.0


## Aggregate events into visits

One row per `(client_id, visit_id)`.

In [13]:
df_visits = df_events.groupby(["client_id", "visit_id"]).agg(
    visit_start = ("date_time", "min"),
    visit_end = ("date_time", "max"),
    max_step_reached = ("step_num", "max"),
    n_steps_session = ("step_num", "count"),
    went_backwards = ("step_direction", lambda x: (x == "backward").any()),
    reached_confirm = ("step_num", lambda x: (x == 4).any())).reset_index()

##  Derived KPIs

`error_rate` and `visit_duration_sec` are computed *after* the agg because they depend on columns the agg just created.


In [17]:
df_visits["visit_duration_sec"] = (df_visits["visit_end"] - df_visits["visit_start"]).dt.total_seconds()

In [18]:
df_visits

,client_id,visit_id,visit_start,visit_end,max_step_reached,n_steps_session,went_backwards,reached_confirm,visit_duration_sec
0,169,749567106_99161211863_557568,2017-04-12 20:19:36,2017-04-12 20:23:09,4,5,False,True,213.0
1,336,649044751_80905125055_554468,2017-06-01 07:26:55,2017-06-01 07:42:43,0,2,False,False,948.0
2,546,731811517_9330176838_94847,2017-06-17 10:03:29,2017-06-17 10:05:42,4,5,False,True,133.0
3,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,4,5,False,True,158.0
4,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,4,5,False,True,377.0
...,...,...,...,...,...,...,...,...,...
159107,9999729,99583652_41711450505_426179,2017-04-05 13:40:49,2017-04-05 13:41:04,1,2,False,False,15.0
159108,9999768,85676722_11636430786_122704,2017-06-03 18:05:10,2017-06-03 18:13:16,4,12,True,True,486.0
159109,9999832,472154369_16714624241_585315,2017-05-16 16:46:03,2017-05-16 16:46:11,1,2,False,False,8.0
159110,9999839,715530422_68620416793_515645,2017-03-29 12:08:55,2017-03-29 12:13:03,4,6,False,True,248.0


##  Save

In [19]:
df_visits.to_csv("visits_kpi_table.csv", index=False)